In [1]:
import pandas as pd
from src.pipelines.BERT_pipeline import BERTPipeline
from src.pipelines.SBERT_pipeline import SBERTPipeline
import logging
import torch
import os

In [2]:
df = pd.read_csv("data/aes_dataset_5k_clean.csv")
df = df[df['dataset'] == 'analisis_essay'][['reference_answer', 'answer', 'score', 'normalized_score', 'dataset', 'dataset_num']]
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 2162 entries, 0 to 2161
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   reference_answer  2162 non-null   object 
 1   answer            2162 non-null   object 
 2   score             2162 non-null   float64
 3   normalized_score  2162 non-null   float64
 4   dataset           2162 non-null   object 
 5   dataset_num       2162 non-null   object 
dtypes: float64(2), object(4)
memory usage: 118.2+ KB
None


,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [3]:
# Check if the first file exists
df_result = None
if os.path.exists("experiments/results/results_sbert.csv"):
    df_result = pd.read_csv("experiments/results/results_sbert.csv")
    print(df_result['config_id'].iloc[-1])
else:
    print("File 'results_sbert.csv' does not exist.")

File 'results_sbert.csv' does not exist.


In [4]:
batch_sizes = [4, 8, 16]
learning_rates = [1e-5, 2e-5, 5e-5, 1e-4]
warm_ups = [0.0, 0.3]
idx = (df_result['config_id'].iloc[-1] + 1) if df_result is not None and not df_result.empty else 0  # index untuk setiap kombinasi
ROOT_DIR = os.getcwd()

In [5]:
for batch_size in batch_sizes:
    for lr in learning_rates:
        for warm_up in warm_ups:
            results = []
            results_epoch = []
            df_result1 = None
            # Check if the second file exists
            if os.path.exists("experiments/results/results_epoch_sbert.csv"):
                df_result1 = pd.read_csv("experiments/results/results_epoch_sbert.csv")
                print(max(df_result1['valid_pearson']))
            else:
                print("File 'results_epoch_sbert.csv' does not exist.")

            # set up hyperparamter
            config = {
                "df": df,
                # "model_name": "indobenchmark/indobert-lite-base-p2",
                "model_name": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
                # "model_name": "all-MiniLM-L6-v2",
                "batch_size": batch_size,
                "learning_rate": lr,
                "epochs": 100,
                "config_id": idx,
                "best_valid_pearson": max(df_result1['valid_pearson']) if df_result1 is not None and not df_result1.empty else float("-inf"),
                "warmup_ratio": warm_up,
            }

            logging.info(
                f"Running configuration: config_id={idx}, model_name={config['model_name']}"
                f", batch_size={batch_size}, epochs={100}, learning_rate={lr}"
            )
            
            print(
                f"\nRunning configuration: config_id={idx}, model_name={config['model_name']}"
                f", batch_size={batch_size}, epochs={100}, learning_rate={lr}"
            )
            
            try:
                pipeline = SBERTPipeline(config, results, results_epoch)
                pipeline.training()

                # Save results
                # Dapatkan root project
                results_path = os.path.join(ROOT_DIR, "experiments/results/results_sbert.csv")
                results_epoch_path = os.path.join(ROOT_DIR, "experiments/results/results_epoch_sbert.csv")
                BERTPipeline.save_csv(results, results_path)
                BERTPipeline.save_csv(results_epoch, results_epoch_path)
            except Exception as e:
                logging.error(f"Error in config_id={idx}: {str(e)}")
                print(f"Error in config_id={idx}: {str(e)}")
                torch.cuda.empty_cache()
            finally:
                # Clear GPU memory after every configuration
                del pipeline.model
                del pipeline.optimizer
                torch.cuda.empty_cache()

            idx += 1

File 'results_epoch_sbert.csv' does not exist.

Running configuration: config_id=0, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=4, epochs=100, learning_rate=1e-05
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======


c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\xlm_roberta\modeling_xlm_roberta.py:371: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/100 - Avg training loss: 0.0294, MAE: 0.1303, RMSE: 0.1715, Pearson Corr: 0.778
Avg validation loss: 0.0242, MAE: 0.1124, RMSE: 0.1561, Pearson Corr: 0.8159
Validation loss decreased (inf --> 0.024174). Saving model ...
Model saved to experiments\models\sentence-transformers/paraphrase-multilingual-mpnet-base-v2_best_model.pt
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0123, MAE: 0.08439, RMSE: 0.1111, Pearson Corr: 0.9135
Avg validation loss: 0.0189, MAE: 0.1, RMSE: 0.1383, Pearson Corr: 0.8669
Validation loss decreased (0.024174 --> 0.018917). Saving model ...
Model saved to experiments\models\sentence-transformers/paraphrase-multilingual-mpnet-base-v2_best_model.pt
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0076, MAE: 0.06637, RMSE: 0.08709, Pearson Corr: 0.9478
Avg validation loss: 0.0160, MAE: 0.08985, RMSE: 0.1273, Pearson Corr: 0.8745
Validation loss decreased (0.018917 --> 0.016009). Saving model ...
Model saved to ex

c:\Users\User\Documents\Code\aes\src\pipelines\SBERT_pipeline.py:122: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('experiments/models/checkpoint.pt

Avg testing loss: 0.0109, MAE: 0.07644, RMSE: 0.1049, Pearson Corr: 0.9135
0.9031340210262172

Running configuration: config_id=1, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=4, epochs=100, learning_rate=1e-05
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======
Epoch 1/100 - Avg training loss: 0.0777, MAE: 0.2375, RMSE: 0.2787, Pearson Corr: 0.2625
Avg validation loss: 0.0775, MAE: 0.2391, RMSE: 0.279, Pearson Corr: 0.4078
Validation loss decreased (inf --> 0.077462). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0665, MAE: 0.2155, RMSE: 0.2579, Pearson Corr: 0.5122
Avg validation loss: 0.0555, MAE: 0.1998, RMSE: 0.2366, Pearson Corr: 0.6089
Validation loss decreased (0.077462 --> 0.055515). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0429, MAE: 0.1677, RMSE: 0.2071, Pearson Corr: 0.7138
Avg va

...

Avg testing loss: 0.0120, MAE: 0.08051, RMSE: 0.1097, Pearson Corr: 0.9066
0.9079274041904536

Running configuration: config_id=22, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=16, epochs=100, learning_rate=0.0001
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======
Epoch 1/100 - Avg training loss: 0.0269, MAE: 0.1247, RMSE: 0.1639, Pearson Corr: 0.7974
Avg validation loss: 0.0252, MAE: 0.112, RMSE: 0.159, Pearson Corr: 0.8065
Validation loss decreased (inf --> 0.025192). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0132, MAE: 0.08721, RMSE: 0.1148, Pearson Corr: 0.9065
Avg validation loss: 0.0213, MAE: 0.1027, RMSE: 0.1468, Pearson Corr: 0.8337
Validation loss decreased (0.025192 --> 0.021320). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0094, MAE: 0.07381, RMSE: 0.09698, Pearson Corr: 0.9341
A

c:\Users\User\Documents\Code\aes\src\pipelines\SBERT_pipeline.py:122: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('experiments/models/checkpoint.pt

Avg testing loss: 0.0118, MAE: 0.08134, RMSE: 0.1092, Pearson Corr: 0.9059
0.9079274041904536

Running configuration: config_id=23, model_name=sentence-transformers/paraphrase-multilingual-mpnet-base-v2, batch_size=16, epochs=100, learning_rate=0.0001
run split dataset...
run split dataset...
create dataset run...
create dataloader run...
====== Training Epoch 1/100 ======
Epoch 1/100 - Avg training loss: 0.0724, MAE: 0.2257, RMSE: 0.269, Pearson Corr: 0.31
Avg validation loss: 0.0547, MAE: 0.1976, RMSE: 0.2355, Pearson Corr: 0.6504
Validation loss decreased (inf --> 0.054746). Saving model ...
====== Training Epoch 2/100 ======
Epoch 2/100 - Avg training loss: 0.0342, MAE: 0.1444, RMSE: 0.1849, Pearson Corr: 0.7529
Avg validation loss: 0.0240, MAE: 0.1155, RMSE: 0.154, Pearson Corr: 0.8119
Validation loss decreased (0.054746 --> 0.024007). Saving model ...
====== Training Epoch 3/100 ======
Epoch 3/100 - Avg training loss: 0.0174, MAE: 0.1009, RMSE: 0.1321, Pearson Corr: 0.8766
Avg va

c:\Users\User\Documents\Code\aes\src\pipelines\SBERT_pipeline.py:122: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('experiments/models/checkpoint.pt

Avg testing loss: 0.0110, MAE: 0.07753, RMSE: 0.1051, Pearson Corr: 0.9127
